In [10]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from alerce.core import Alerce
import numpy as np

alerce = Alerce()

In [11]:
ZTF_IDS = [
    "ZTF22abhwlnm", "ZTF22aadesjc", "ZTF22abfdzrv", "ZTF21abgkfzh",
    "ZTF23aatcsou", "ZTF23aadjssg", "ZTF18aaiwzie", "ZTF23aaekebt",
    "ZTF23aadbtou", "ZTF22abghrui", "ZTF21abiggqx", "ZTF23aaefpfb",
    "ZTF22aatwxrl", "ZTF23aaahnss", "ZTF22absuavp", "ZTF23aafgmaz",
    "ZTF23aaflptz", "ZTF23aatcola", "ZTF23aajestr", "ZTF23aahjdxa",
    "ZTF22abrbohu", "ZTF23aarzzwu", "ZTF22abzajwl", "ZTF23aagxvad", "ZTF23aaempzk"
]

SUPERNOVA_CLASSES = ['sn', 'snia', 'snibc', 'snii', 'slsn']


In [12]:
def check_supernova(obj_id):
    df = alerce.query_probabilities(oid=obj_id, format='pandas')
    
    lc = df[df['classifier_name'].str.contains('lc_classifier')]
    stamp = df[df['classifier_name'].str.contains('stamp_classifier')]
    
    lc_sn = lc[lc['class_name'].str.lower().isin(SUPERNOVA_CLASSES)]
    stamp_sn = stamp[stamp['class_name'].str.lower().isin(SUPERNOVA_CLASSES)]
    
    result = {'oid': obj_id}
    
    if not lc_sn.empty:
        lc_top = lc_sn.loc[lc_sn['probability'].idxmax()]
        result['lc_class'] = lc_top['class_name']
        result['lc_prob'] = lc_top['probability']
    else:
        result['lc_class'] = None
        result['lc_prob'] = None

    if not stamp_sn.empty:
        stamp_top = stamp_sn.loc[stamp_sn['probability'].idxmax()]
        result['stamp_class'] = stamp_top['class_name']
        result['stamp_prob'] = stamp_top['probability']
    else:
        result['stamp_class'] = None
        result['stamp_prob'] = None

    # Determine if object is a supernova
    if lc_sn.empty and stamp_sn.empty:
        top = df.loc[df['probability'].idxmax()]
        result.update({
            'is_supernova': False,
            'agreement': None,
            'probability': top['probability'],
            'most_likely_class': top['class_name']
        })
    else:
        result.update({
            'is_supernova': True,
            'agreement': (result['lc_class'] == result['stamp_class']) if result['lc_class'] and result['stamp_class'] else None,
            'probability': ((result['lc_prob'] or 0) + (result['stamp_prob'] or 0)) / (2 if result['lc_prob'] and result['stamp_prob'] else 1),
            'most_likely_class': None
        })
    
    return result

In [13]:
def build_features_dataframe(sn_ids):
    data_list = []
    for oid in sn_ids:
        try:
            feats = alerce.query_features(oid, format="pandas")
            if feats is not None and not feats.empty:
                feat_dict = feats.set_index("name")["value"].to_dict()
                feat_dict["oid"] = oid
                data_list.append(feat_dict)
        except Exception as e:
            print(f"Error fetching features for {oid}: {e}")
    return pd.DataFrame(data_list)

In [14]:
def map_features_to_physical(df):
    mapped = df.copy()
    rename_dict = {
        "mag_max": "mag_max",
        "t_rise": "t_rise",
        "dm15": "dm15",
        "Mean": "Mean",
        "g-r_mean": "g-r_mean",
        "MaxSlope": "MaxSlope"
    }
    mapped.rename(columns={old: new for old, new in rename_dict.items() if old in df.columns}, inplace=True)
    return mapped

In [15]:
OUTLIER_THRESHOLD = 0.2
def compute_difference_and_flag(df, reference_props):
    diff_table = df.copy()
    for prop, ref_value in reference_props.items():
        if prop in df.columns:
            diff_table[f"{prop}_diff"] = df[prop] - ref_value
            diff_table[f"{prop}_outlier"] = diff_table[f"{prop}_diff"].abs() > OUTLIER_THRESHOLD
    return diff_table

def extract_outliers(diff_table):
    outlier_cols = [col for col in diff_table.columns if col.endswith("_outlier")]
    diff_cols = [col for col in diff_table.columns if col.endswith("_diff")]
    outlier_table = diff_table.copy()
    mask = outlier_table[outlier_cols].any(axis=1)
    outlier_table = outlier_table.loc[mask, ["oid"] + diff_cols + outlier_cols]
    return outlier_table

In [16]:
def plot_comparison(df, typeIa_avg, properties=["mag_max","t_rise"]):
    for prop in properties:
        if prop in df.columns:
            plt.figure(figsize=(6,4))
            plt.scatter(df['oid'], df[prop], s=80, color='blue', label='Your SNs')
            plt.axhline(typeIa_avg[prop], color='red', linestyle='--', label='Type Ia Average')
            plt.xticks(rotation=45)
            plt.ylabel(prop)
            plt.title(f'{prop} Comparison to Type Ia Average')
            plt.legend()
            plt.tight_layout()
            plt.show()

In [17]:
def fetch_typeIa_reference(n_samples=20):
    try:
        typeIa_sn = alerce.query_objects(
            classifier="lc_classifier",
            class_name="SNIa",
            probability_threshold=0.7,
            format="pandas"
        ).head(n_samples)

        feats_list = []
        for oid in typeIa_sn['oid']:
            feats = alerce.query_features(oid, format="pandas")
            if feats is not None and not feats.empty:
                feat_dict = feats.set_index("name")["value"].to_dict()
                feat_dict["oid"] = oid
                feats_list.append(feat_dict)
        return pd.DataFrame(feats_list) if feats_list else pd.DataFrame()
    except Exception as e:
        print("Error fetching Type Ia SNe:", e)
        return pd.DataFrame()

In [18]:
def main():
    # Step 6a: Classification
    classification_results = [check_supernova(oid) for oid in ZTF_IDS]
    class_df = pd.DataFrame(classification_results)
    
    # Step 6b: Features
    feature_df = build_features_dataframe(ZTF_IDS)
    if feature_df.empty:
        print("No features available for your input SNs.")
        return
    feature_df = map_features_to_physical(feature_df)
    
    # Keep only relevant features
    feature_cols = ["oid", "mag_max", "t_rise", "dm15", "Mean", "g-r_mean", "MaxSlope"]
    feature_df = feature_df[[col for col in feature_cols if col in feature_df.columns]]
    
    # Merge classification + features
    combined_df = pd.merge(class_df, feature_df, on="oid", how="left")
    
    display(Markdown("## Supernova Features + Classification Summary"))
    pd.set_option('display.max_columns', None)
    display(combined_df)
    
    # Step 6c: Type Ia comparison
    typeIa_df = fetch_typeIa_reference(n_samples=20)
    if typeIa_df.empty:
        print("Cannot compute Type Ia reference differences. Exiting.")
        return
    typeIa_mapped = map_features_to_physical(typeIa_df)
    
    # Only keep numeric columns that exist
    numeric_cols = ["mag_max", "t_rise", "dm15", "Mean", "g-r_mean", "MaxSlope"]
    existing_cols = [col for col in numeric_cols if col in typeIa_mapped.columns]
    typeIa_avg_filtered = typeIa_mapped[existing_cols].select_dtypes(include=np.number).mean()
    
    display(Markdown("## Typical Type Ia Average Values"))
    display(pd.DataFrame(typeIa_avg_filtered).T)
    
    # Differences & outliers
    diff_table = compute_difference_and_flag(feature_df, typeIa_avg_filtered)
    display(Markdown("## Differences from Type Ia & Outlier Flags"))
    display(diff_table)
    
    outliers = extract_outliers(diff_table)
    display(Markdown("## Only Outlier Features"))
    if not outliers.empty:
        display(outliers)
    else:
        display(Markdown("No outliers detected."))
    
    # Plot comparison
    plot_comparison(feature_df, typeIa_avg_filtered, properties=["mag_max","t_rise"])

# Run the workflow
main()

## Supernova Features + Classification Summary

,oid,lc_class,lc_prob,stamp_class,stamp_prob,is_supernova,agreement,probability,most_likely_class,Mean,g-r_mean,MaxSlope
0,ZTF22abhwlnm,SLSN,0.650,SN,0.544367,True,False,0.597183,None,NaN,0.100957,NaN
1,ZTF22aadesjc,SLSN,0.680,SN,0.732096,True,False,0.706048,None,18.463453,0.311640,0.867480
2,ZTF22abfdzrv,SLSN,0.724,SN,0.313880,True,False,0.518940,None,19.629977,0.285459,0.067661
3,ZTF21abgkfzh,SLSN,0.418,SN,0.566296,True,False,0.492148,None,NaN,0.459420,NaN
4,ZTF23aatcsou,SLSN,0.614,SN,0.334540,True,False,0.474270,None,19.820835,0.190543,0.120993
5,ZTF23aadjssg,SNIbc,0.334,SN,0.716764,True,False,0.525382,None,18.454111,1.046436,0.081238
6,ZTF18aaiwzie,SNIa,0.416,SN,0.553450,True,False,0.484725,None,18.430837,-0.392514,88.263345
7,ZTF23aaekebt,SNII,0.510,SN,0.391327,True,False,0.450663,None,18.922177,0.356721,256.313378
8,ZTF23aadbtou,SNII,0.366,SN,0.665639,True,False,0.515820,None,18.848800,0.143260,0.118470
9,ZTF22abghrui,SLSN,0.648,SN,0.252840,True,False,0.450420,None,19.354617,0.356615,177.550690


## Typical Type Ia Average Values

,Mean,g-r_mean,MaxSlope
0,16.945455,-0.581162,19.000796


## Differences from Type Ia & Outlier Flags

,oid,Mean,g-r_mean,MaxSlope,Mean_diff,Mean_outlier,g-r_mean_diff,g-r_mean_outlier,MaxSlope_diff,MaxSlope_outlier
0,ZTF22abhwlnm,NaN,0.100957,NaN,NaN,False,0.682119,True,NaN,False
1,ZTF22aadesjc,18.463453,0.311640,0.867480,1.517998,True,0.892802,True,-18.133315,True
2,ZTF22abfdzrv,19.629977,0.285459,0.067661,2.684523,True,0.866621,True,-18.933135,True
3,ZTF21abgkfzh,NaN,0.459420,NaN,NaN,False,1.040582,True,NaN,False
4,ZTF23aatcsou,19.820835,0.190543,0.120993,2.875380,True,0.771705,True,-18.879802,True
5,ZTF23aadjssg,18.454111,1.046436,0.081238,1.508656,True,1.627598,True,-18.919557,True
6,ZTF18aaiwzie,18.430837,-0.392514,88.263345,1.485382,True,0.188649,False,69.262550,True
7,ZTF23aaekebt,18.922177,0.356721,256.313378,1.976722,True,0.937884,True,237.312583,True
8,ZTF23aadbtou,18.848800,0.143260,0.118470,1.903345,True,0.724422,True,-18.882326,True
9,ZTF22abghrui,19.354617,0.356615,177.550690,2.409163,True,0.937777,True,158.549895,True


## Only Outlier Features

,oid,Mean_diff,g-r_mean_diff,MaxSlope_diff,Mean_outlier,g-r_mean_outlier,MaxSlope_outlier
0,ZTF22abhwlnm,NaN,0.682119,NaN,False,True,False
1,ZTF22aadesjc,1.517998,0.892802,-18.133315,True,True,True
2,ZTF22abfdzrv,2.684523,0.866621,-18.933135,True,True,True
3,ZTF21abgkfzh,NaN,1.040582,NaN,False,True,False
4,ZTF23aatcsou,2.875380,0.771705,-18.879802,True,True,True
5,ZTF23aadjssg,1.508656,1.627598,-18.919557,True,True,True
6,ZTF18aaiwzie,1.485382,0.188649,69.262550,True,False,True
7,ZTF23aaekebt,1.976722,0.937884,237.312583,True,True,True
8,ZTF23aadbtou,1.903345,0.724422,-18.882326,True,True,True
9,ZTF22abghrui,2.409163,0.937777,158.549895,True,True,True
